<a href="https://colab.research.google.com/github/SivaSwetha-baba/my_first_repo/blob/main/support_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required libraries
!pip uninstall -y langchain langchain-core langgraph
!pip install -qqq chromadb==0.5.3 sentence-transformers==2.7.0 langgraph==1.2.12 pydantic==2.8.2 "uvicorn[standard]"==0.30.1 fastapi==0.111.0 'numpy<2.0.0' langchain==1.3.18 langchain-core==1.6.0

import json
import os
import re
from pathlib import Path
from typing import Any, Literal, TypedDict

import chromadb
from fastapi import FastAPI
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field, ValidationError
from sentence_transformers import SentenceTransformer

BASE_DIR = Path("/content").resolve() # Modified: Use /content as base directory for Colab
CHROMA_DIR = BASE_DIR / "chroma_db"
COLLECTION_NAME = "zepto_policies"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

POLICY_KEYWORDS = (
    "delivery", "return", "refund", "membership",
    "tracking", "cancel", "gift card", "support hours",
)

STRUCTURED_PROMPT = """ROLE:
You are a Zepto policy assistant. You answer only from the supplied Zepto policy context.

CONTEXT:
{context}

TASK:
Answer the user's question using only the policy facts in CONTEXT. If the context does not contain enough information, say that the provided context does not specify the answer.

FORMAT:
Return valid JSON with exactly these fields:
{{"answer": "string", "sources": ["chunk/document IDs"], "confidence": 0.0}}
Do not use markdown fences. The sources list must contain only IDs that actually occur in CONTEXT.

LENGTH:
Keep the answer concise, normally 1-3 sentences.

NEGATIVE CONSTRAINT:
Do not answer using information that is not present in the provided context. Do not invent Zepto policies, prices, dates, limits, or exceptions.

FEW-SHOT EXAMPLE:
User question: "How much is priority delivery?"
Context: "Priority delivery is available at checkout for an additional INR 15."
Output: {{"answer":"Priority delivery costs an additional INR 15.","sources":["doc_01_chunk_01"],"confidence":1.0}}

USER QUESTION:
{question}
"""

CLASSIFY_PROMPT = """ROLE:
You are an intent classifier for a Zepto policy service.

CONTEXT:
The policy corpus covers Zepto delivery, returns/refunds, membership, tracking, cancellation, gift cards, and customer support hours.

TASK:
Classify the user's query as exactly one of: policy_question or general_question.

FORMAT:
Return valid JSON exactly like {{"intent":"policy_question"}} or {{"intent":"general_question"}}.

LENGTH:
Return JSON only, with no explanation.

NEGATIVE CONSTRAINT:
Do not invent an intent category. Do not answer the user's question.

FEW-SHOT EXAMPLES:
User: "What is the delivery fee?"
Output: {{"intent":"policy_question"}}
User: "What is the capital of France?"
Output: {{"intent":"general_question"}}

USER:
{question}
"""

DIRECT_PROMPT = """ROLE:
You are an Zepto assistant.

CONTEXT:
No Zepto policy retrieval context is available because this query was classified as general_question.

TASK:
Answer the user's question directly.

FORMAT:
Return valid JSON with fields answer, sources, confidence.

LENGTH:
Keep it concise.

NEGATIVE CONSTRAINT:
Do not pretend that a general answer is a Zepto policy answer, and do not fabricate Zepto policy details.

FEW-SHOT EXAMPLE:
User: "What is 2 + 2?"
Output: {{"answer":"4","sources":[],"confidence":1.0}}

USER:
{question}
"""


class AskRequest(BaseModel):
    query: str = Field(min_length=1)


class AnswerResponse(BaseModel):
    answer: str
    sources: list[str]
    confidence: float = Field(ge=0.0, le=1.0)


class GraphState(TypedDict, total=False):
    query: str
    intent: Literal["policy_question", "general_question"]
    retrieved: list[dict[str, Any]]
    response: dict[str, Any]


_embedding_model = None
_collection = None

def mock_enabled() -> bool:
    return os.getenv("MOCK_LLM", "1") != "0"

def get_embedding_model():
    global _embedding_model
    if _embedding_model is None:
        _embedding_model = SentenceTransformer(EMBEDDING_MODEL)
    return _embedding_model

def get_collection():
    global _collection
    if _collection is None:
        client = chromadb.PersistentClient(path=str(CHROMA_DIR))
        _collection = client.get_collection(COLLECTION_NAME)
    return _collection


def classify_intent(state: GraphState) -> GraphState:
    query = state["query"]
    if mock_enabled():
        q = query.lower()
        intent = (
            "policy_question"
            if any(keyword in q for keyword in POLICY_KEYWORDS)
            else "general_question"
        )
        return {"intent": intent}

    raw = call_real_llm(
        CLASSIFY_PROMPT.format(question=query),
        max_retries=2,
        schema="intent",
    )
    intent = raw.get("intent")
    if intent not in {"policy_question", "general_question"}:
        raise ValueError("LLM returned an invalid intent")
    return {"intent": intent}


def retrieve_top3(query: str) -> list[dict[str, Any]]:
    model = get_embedding_model()
    embedding = model.encode([query], normalize_embeddings=True).tolist()
    result = get_collection().query(
        query_embeddings=embedding,
        n_results=3,
        include=["documents", "metadatas", "distances"],
    )
    rows = []
    for document, metadata, distance in zip(
        result["documents"][0],
        result["metadatas"][0],
        result["distances"][0],
    ):
        rows.append(
            {
                "id": metadata["chunk_id"],
                "document_id": metadata["document_id"],
                "source": metadata["source"],
                "content": document,
                "distance": float(distance),
            }
        )
    return rows


def retrieve_and_answer(state: GraphState) -> GraphState:
    query = state["query"]
    retrieved = retrieve_top3(query)

    if mock_enabled():
        top_snippet = retrieved[0]["content"][:200]
        response = AnswerResponse(
            answer=f"Based on the retrieved context: {top_snippet}",
            sources=[item["id"] for item in retrieved],
            confidence=1.0,
        )
        return {"retrieved": retrieved, "response": response.model_dump()}

    context = "\n\n".join(
        f"ID: {item['id']}\nCONTENT: {item['content']}"
        for item in retrieved
    )
    raw = call_real_llm(
        STRUCTURED_PROMPT.format(context=context, question=query),
        max_retries=2,
        schema="answer",
    )
    raw["sources"] = [item["id"] for item in retrieved]
    response = AnswerResponse.model_validate(raw)
    return {"retrieved": retrieved, "response": response.model_dump()}


def direct_answer(state: GraphState) -> GraphState:
    if mock_enabled():
        response = AnswerResponse(
            answer="I can only answer questions about Zepto policies right now.",
            sources=[],
            confidence=1.0,
        )
        return {"response": response.model_dump()}

    raw = call_real_llm(
        DIRECT_PROMPT.format(question=state["query"]),
        max_retries=2,
        schema="answer",
    )
    raw["sources"] = []
    response = AnswerResponse.model_validate(raw)
    return {"response": response.model_dump()}


def route_after_classify(state: GraphState) -> str:
    return state["intent"]


def extract_json(text: str) -> dict[str, Any]:
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.I)
    match = re.search(r"\{.*\}", text, flags=re.S)
    if not match:
        raise ValueError("No JSON object found")
    return json.loads(match.group(0))


def get_llm():
    from langchain_groq import ChatGroq

    api_key = os.getenv("GROQ_API_KEY")
    if not api_key:
        raise RuntimeError("Set GROQ_API_KEY when MOCK_LLM=0.")
    return ChatGroq(
        model=os.getenv("GROQ_MODEL", "llama-3.1-8b-instant"),
        temperature=0,
        api_key=api_key,
    )


def call_real_llm(
    prompt: str,
    max_retries: int = 2,
    schema: str = "answer",
) -> dict[str, Any]:
    llm = get_llm()
    corrective = ""

    for attempt in range(max_retries + 1):
        result = llm.invoke(prompt + corrective)
        content = result.content if hasattr(result, "content") else str(result)

        try:
            data = extract_json(content)
            if schema == "intent":
                if data.get("intent") not in {
                    "policy_question",
                    "general_question",
                }:
                    raise ValidationError.from_exception_data(
                        "Intent", []
                    )
                return data

            return AnswerResponse.model_validate(data).model_dump()

        except (ValidationError, ValueError, json.JSONDecodeError):
            if attempt == max_retries:
                return {
                    "answer": "ERROR: the real LLM response failed schema validation after 3 attempts.",
                    "sources": [],
                    "confidence": 0.0,
                }

            corrective = (
                "\n\nCORRECTIVE INSTRUCTION: Your previous response failed "
                "schema validation. Return ONLY valid JSON matching the "
                "required schema. No markdown or commentary."
            )

    raise RuntimeError("Unreachable")


def build_graph():
    builder = StateGraph(GraphState)
    builder.add_node("classify_intent", classify_intent)
    builder.add_node("retrieve_and_answer", retrieve_and_answer)
    builder.add_node("direct_answer", direct_answer)

    builder.add_edge(START, "classify_intent")
    builder.add_conditional_edges(
        "classify_intent",
        route_after_classify,
        {
            "policy_question": "retrieve_and_answer",
            "general_question": "direct_answer",
        },
    )
    builder.add_edge("retrieve_and_answer", END)
    builder.add_edge("direct_answer", END)
    return builder.compile()

graph = build_graph()
app = FastAPI(title="Zepto GenAI Policy Service", version="1.0.0")


@app.get("/health")
def health():
    return {"status": "ok", "mock_llm": mock_enabled()}


@app.post("/ask", response_model=AnswerResponse)
def ask(request: AskRequest) -> AnswerResponse:
    result = graph.invoke({"query": request.query})
    return AnswerResponse.model_validate(result["response"])

Found existing installation: langchain 1.3.18
Uninstalling langchain-1.3.18:
  Successfully uninstalled langchain-1.3.18
Found existing installation: langchain-core 1.6.0
Uninstalling langchain-core-1.6.0:
  Successfully uninstalled langchain-core-1.6.0
Found existing installation: langgraph 1.2.12
Uninstalling langgraph-1.2.12:
  Successfully uninstalled langgraph-1.2.12


In [ ]:
# Temporarily install langchain ecosystem components separately for debugging dependency resolution.
!pip uninstall -y langchain langchain-core langgraph

# Install all dependencies in a single command to ensure consistent version resolution
!pip install chromadb==0.5.3 sentence-transformers==2.7.0 pydantic==2.8.2 "uvicorn[standard]"==0.30.1 fastapi==0.111.0 'numpy<2.0.0' langchain==1.3.18 langchain-core==1.6.0 langgraph==1.2.12

In [ ]:
!pip3 install chromadb==0.5.3 sentence-transformers==2.7.0 langgraph==0.0.40 pydantic==2.8.2 "uvicorn[standard]"==0.30.1 fastapi==0.111.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.2/125.2 kB 5.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 84.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of langchain-core to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-core to determine which version is compatible with other requirements. Th

In [ ]:
from pathlib import Path

import chromadb
from sentence_transformers import SentenceTransformer

BASE_DIR = Path("/content").resolve() # Modified: Use /content as base directory for Colab
DOCS_DIR = BASE_DIR / "docs"
CHROMA_DIR = BASE_DIR / "chroma_db"
COLLECTION_NAME = "zepto_policies"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"


def load_documents():
    documents = []
    ids = []
    metadatas = []

    for path in sorted(DOCS_DIR.glob("doc_*.txt")):
        text = path.read_text(encoding="utf-8").strip()
        chunk_id = f"{path.stem}_chunk_01"
        documents.append(text)
        ids.append(chunk_id)
        metadatas.append(
            {
                "document_id": path.stem,
                "chunk_id": chunk_id,
                "source": path.name,
            }
        )

    return documents, ids, metadatas


def main():
    documents, ids, metadatas = load_documents()

    model = SentenceTransformer(EMBEDDING_MODEL)
    embeddings = model.encode(
        documents,
        normalize_embeddings=True,
    ).tolist()

    client = chromadb.PersistentClient(path=str(CHROMA_DIR))

    try:
        client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass

    collection = client.get_or_create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"},
    )

    collection.add(
        ids=ids,
        documents=documents,
        metadatas=metadatas,
        embeddings=embeddings,
    )

    print(f"Indexed {len(documents)} documents into '{COLLECTION_NAME}'.")
    print("IDs:", ids)


if __name__ == "__main__":
    main()


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

ValueError: Could not find BertModel neither in <module 'transformers.models.bert' from '/usr/local/lib/python3.13/dist-packages/transformers/models/bert/__init__.py'> nor in <module 'transformers' from '/usr/local/lib/python3.13/dist-packages/transformers/__init__.py'>!